# Graph Construction

## Objective

The objective of this notebook is to transform the OpenFlights airport and route datasets into a directed graph representation using NetworkX. Airports are modeled as nodes, while direct flight routes are modeled as directed edges. The resulting graph will serve as the foundation for all subsequent network analysis.

## Import Libraries

In [1]:
import pandas as pd
import networkx as nx

## Load Datasets

In [2]:
airports = pd.read_csv("../data/airports.dat", header=None)
routes = pd.read_csv("../data/routes.dat", header=None)

Rename Coloumns for Airports and Routes

In [3]:
airports.columns = [
    "Airport_ID",
    "Airport_Name",
    "City",
    "Country",
    "IATA",
    "ICAO",
    "Latitude",
    "Longitude",
    "Altitude",
    "Timezone",
    "DST",
    "Tz_Database_Timezone",
    "Type",
    "Source"
]

In [4]:
routes.columns = [
    "Airline",
    "Airline_ID",
    "Source_Airport",
    "Source_Airport_ID",
    "Destination_Airport",
    "Destination_Airport_ID",
    "Codeshare",
    "Stops",
    "Equipment"
]

In [5]:
print("Airport Dataset Shape:", airports.shape)
print("Route Dataset Shape:", routes.shape)

Airport Dataset Shape: (7698, 14)
Route Dataset Shape: (67663, 9)


## Constructing the Airport Network

### Construction Strategy

The airport transportation network is modeled as a **directed graph**, where:

- Each airport is represented as a node.
- Each scheduled flight route is represented as a directed edge.
- A route from Airport A to Airport B is treated as different from a route from Airport B to Airport A, allowing the network to preserve the direction of air traffic.

Before adding routes, all airports are first inserted as graph nodes. Routes are then added only when both the source and destination airports exist in the dataset, ensuring that the resulting graph remains internally consistent.

In [6]:
# Create an empty directed graph
G = nx.DiGraph()

In [7]:
# Add all airports as graph nodes
for _, row in airports.iterrows():

    airport_id = int(row["Airport_ID"])

    G.add_node(
        airport_id,
        name=row["Airport_Name"],
        city=row["City"],
        country=row["Country"],
        iata=row["IATA"],
        latitude=row["Latitude"],
        longitude=row["Longitude"]
    )

In [8]:
print("Nodes after adding airports:", G.number_of_nodes())

Nodes after adding airports: 7698


In [9]:
# Valid airport IDs
airport_ids = set(airports["Airport_ID"].astype(int))

edges_added = 0
edges_skipped = 0

for _, row in routes.iterrows():

    source = row["Source_Airport_ID"]
    destination = row["Destination_Airport_ID"]

    # Skip missing airport IDs
    if source == "\\N" or destination == "\\N":
        edges_skipped += 1
        continue

    source = int(source)
    destination = int(destination)

    # Only connect airports present in the airport dataset
    if source in airport_ids and destination in airport_ids:
        G.add_edge(source, destination)
        edges_added += 1
    else:
        edges_skipped += 1

In [10]:
print("Number of Nodes :", G.number_of_nodes())
print("Number of Edges :", G.number_of_edges())
print("Routes Processed :", len(routes))
print("Routes Added :", edges_added)
print("Routes Skipped :", edges_skipped)

Number of Nodes : 7698
Number of Edges : 36907
Routes Processed : 67663
Routes Added : 66771
Routes Skipped : 892


## Observations

- The airport dataset contains 7,698 airports.
- After validating route records, 66,771 valid routes were processed.
- 892 routes were discarded because they contained missing or invalid airport IDs.
- The resulting graph contains 36,907 unique directed edges because NetworkX DiGraph stores only one edge between any ordered pair of airports, even if multiple airlines operate the same route.
- The constructed graph will be used for all subsequent network analysis.

In [11]:
import networkx as nx
print(nx.__version__)

3.6.1


Note:-For subsequent analysis, this graph construction has been modularized into src/graph_builder.py.

## Conclusion

This notebook transforms the raw airport and route datasets into a directed graph representation suitable for network analysis.

By modeling airports as nodes and flight routes as directed edges, the resulting graph preserves both the connectivity and directionality of global air transportation. Careful validation during graph construction ensures that only valid airport-to-airport connections are included.

The constructed network serves as the foundation for all subsequent analyses in this project, including network statistics, centrality measures, geographic visualization, community detection, and resilience analysis.